In [2]:
import boto3
from botocore.exceptions import ClientError
from urllib.parse import urlparse
from pathlib import Path
import posixpath

TARGET_SUBFOLDER = "target"

def parse_s3_uri(s3_uri: str):
    """Return (bucket, key_prefix) from an s3:// URI. key_prefix ends with '/' or ''."""
    parsed = urlparse(s3_uri)
    if parsed.scheme != "s3":
        raise ValueError(f"Expected an s3:// URI, got: {s3_uri}")
    bucket = parsed.netloc
    key = parsed.path.lstrip('/')  # may be '' or 'some/prefix'
    if key and not key.endswith('/'):
        key += '/'
    return bucket, key

def build_target_prefix(base_prefix: str, subfolder: str = TARGET_SUBFOLDER) -> str:
    """
    Make the full Prefix that points to the subfolder.
    If base_prefix already ends with the subfolder, keep it as-is.
    """
    base_norm = base_prefix.rstrip('/')
    if base_norm.endswith('/' + subfolder) or base_norm == subfolder:
        full = base_norm + '/'
    else:
        full = posixpath.join(base_prefix, subfolder) + '/'
    return full

def download_target_final_annotations(
        s3_folder_uri: str,
        local_dir: str = "./s3_downloads",
        profile_name: str | None = None,
        skip_existing: bool = True,
):
    """
    Download all objects under {s3_folder_uri}/target_final_annotations/ recursively into local_dir,
    preserving the S3 subdirectory structure.
    """
    bucket, base_prefix = parse_s3_uri(s3_folder_uri)
    sub_prefix = build_target_prefix(base_prefix, TARGET_SUBFOLDER)

    session = boto3.Session(profile_name=profile_name) if profile_name else boto3.Session()
    s3 = session.client("s3")
    paginator = s3.get_paginator("list_objects_v2")

    local_root = Path(local_dir)
    local_root.mkdir(parents=True, exist_ok=True)

    downloaded = skipped = 0
    any_pages = False

    for page in paginator.paginate(Bucket=bucket, Prefix=sub_prefix):
        any_pages = True
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith('/'):  # skip "folders"
                continue

            rel_path = key[len(sub_prefix):]  # path relative to the target subfolder
            dest_path = local_root / rel_path
            dest_path.parent.mkdir(parents=True, exist_ok=True)

            if skip_existing and dest_path.exists():
                try:
                    if dest_path.stat().st_size == obj.get("Size"):
                        skipped += 1
                        continue
                except OSError:
                    pass

            try:
                s3.download_file(bucket, key, str(dest_path))
                downloaded += 1
            except ClientError as e:
                print(f"Failed to download {key}: {e}")

    if not any_pages or (downloaded == 0 and skipped == 0):
        print(f"No objects found at s3://{bucket}/{sub_prefix}")
    else:
        print(f"Done. Downloaded: {downloaded}, skipped existing: {skipped}. Saved to: {local_root.resolve()}")

# ---- Example usage in a Jupyter cell ----
# base_uri = "s3://my-bucket/path/to/folder/"   # replace with your S3 folder URI
# download_target_final_annotations(base_uri, local_dir="./target_final_annotations_download")

base_uri = "s3://legal-cases-bucket240797/"

download_target_final_annotations(base_uri, local_dir="./target_final_annotations_download")


Done. Downloaded: 71, skipped existing: 0. Saved to: /home/lbrenap/Documents/projects/legalpaca/preprocessing/target_final_annotations_download


In [5]:
import numpy as np

# Example with 1D arrays
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
stacked_1d = np.vstack(a)
print("Stacked 1D arrays:")
print(stacked_1d)

Stacked 1D arrays:
[[1]
 [2]
 [3]]
